# VYTRIX: Intelligent Spatial Engine & Photorealistic 3D Digital Twin

**Team:** VYTRO SQUAD  
**Competition:** Techfest 2026-27, IIT Bombay (Round 1)  
**Repository:** [https://github.com/coder-mrjk/vytrix.git](https://github.com/coder-mrjk/vytrix.git)  

---

## Overview
This official driver notebook executes the complete **VYTRIX 6-Stage Reconstruction Pipeline** on a free **Google Colab T4 GPU**:

1. **Stage 1: Preprocess** (Adaptive decimation & Laplacian blur filtering)
2. **Stage 2: Structure-from-Motion** (COLMAP sequential feature matching + loop closure)
3. **Stage 3: 3D Gaussian Splatting** (GPU rasterizer training with Google Drive checkpoints)
4. **Stage 4: Canonical Normalization** (RANSAC floor fitting, $+Y$ up, $Y=0$, metric scale)
5. **Stage 5: Optimization & Compression** (Opacity pruning & `.ksplat` compression)
6. **Stage 6: Packaging** (Canonical `scene.json` manifest & web asset bundle)

> **Note for Judges:** Ensure this notebook is running on a GPU runtime (**Runtime $\rightarrow$ Change runtime type $\rightarrow$ T4 GPU**).

### Step 1: GPU & Hardware Verification
Verify that an NVIDIA CUDA-enabled GPU (T4, V100, or A100) is active.

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "ERROR: CUDA GPU is not available! Change runtime type to T4 GPU."
print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### Step 2: Install System & Python Dependencies
Install COLMAP for Structure-from-Motion and pinned Python libraries.

In [ ]:
# Install COLMAP
!apt-get update -qq && apt-get install -y -qq colmap ffmpeg

# Install Python dependencies
!pip install -q pyyaml==6.0.1 "numpy>=1.24.0,<2.0.0" scipy>=1.11.0 opencv-python-headless>=4.8.0 pytest>=7.4.0

# Install Apache-2.0 gsplat rasterizer
!pip install -q gsplat

print("All VYTRIX dependencies installed successfully!")

### Step 3: Mount Google Drive (Persistent Checkpoints)
Mounting Google Drive protects your training progress against Colab disconnections.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
checkpoint_dir = '/content/drive/MyDrive/VYTRIX_Checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Google Drive checkpoint directory ready: {checkpoint_dir}")

### Step 4: Clone the VYTRIX Codebase

In [ ]:
%cd /content
!if [ -d "vytrix" ]; then rm -rf vytrix; fi
!git clone https://github.com/coder-mrjk/vytrix.git
%cd /content/vytrix

### Step 5: Input Video Ingestion
Upload your smartphone capture video (following `docs/SOP.md`), or use our built-in sample data.

In [ ]:
import os
from google.colab import files

video_path = "test/data/sample_video.mp4"

USE_UPLOADED_VIDEO = False  # Set to True if you wish to upload your own video

if USE_UPLOADED_VIDEO:
    print("Please select your smartphone video file (MP4)...")
    uploaded = files.upload()
    video_name = list(uploaded.keys())[0]
    video_path = os.path.abspath(video_name)
    print(f"Uploaded video: {video_path}")
else:
    print(f"Using default sample dataset video: {video_path}")

### Step 6: Execute the Full 6-Stage VYTRIX Reconstruction Pipeline
Runs preprocessing, COLMAP SfM, 3DGS training, RANSAC gravity leveling, and packaging.

In [ ]:
import sys
sys.path.insert(0, "pipeline")

# Execute pipeline CLI runner
!PYTHONPATH=pipeline python -m vytrix.run \
    --video "$video_path" \
    --config "config.yaml" \
    --work-dir "data/work"

### Step 7: Download Web Viewer Scene Bundle
Zips the generated digital twin (`scene.json` and `.ksplat` / `.splat` files) and downloads it directly to your computer.

In [ ]:
import shutil
from google.colab import files

output_scene_dir = "viewer/public/scenes/home-01"
bundle_zip = "vytrix_scene_bundle.zip"

shutil.make_archive("vytrix_scene_bundle", "zip", output_scene_dir)
print(f"Generated scene bundle: {bundle_zip}")

# Trigger browser download
files.download(bundle_zip)
print("Downloaded! Unzip and place these files into 'viewer/public/scenes/home-01/' on your local machine to view.")